In [ ]:
#Setting up the environment
%pip install --upgrade openai python-dotenv pandas

In [1]:
#Import libraries
import os
import json
import time
import uuid
import logging
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
#Configuring the API Client
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is missing.")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized.")

OpenAI client initialized.


In [3]:
#Create Project folders

BASE_DIR = Path(".")

PROMPT_DIR = BASE_DIR / "prompts"
DATASET_DIR = BASE_DIR / "datasets"
RESULT_DIR = BASE_DIR / "results"

for folder in [PROMPT_DIR, DATASET_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created.")

Project folders created.


# Creating a Prompt registry

In [4]:
#Define Prompts
prompts = {
    "v1": """
You are a helpful AI assistant.
Answer the user's question.
""",

    "v2": """
You are a reliable AI assistant.

Use the supplied context to answer questions.
Use simple and clear English.
Do not invent information.
If the context does not contain the answer,
say that you do not have enough information.
"""
}

In [5]:
#Save Prompts
PROMPT_FILE = PROMPT_DIR / "prompts.json"

with open(PROMPT_FILE, "w", encoding="utf-8") as file:
    json.dump(prompts, file, indent=4)

print("Prompts saved.")

Prompts saved.


In [6]:
#Select a Prompt Version
ACTIVE_PROMPT_VERSION = "v2"

def get_prompt(version):

    if version not in prompts:
        raise ValueError("Unknown prompt version.")

    return prompts[version]


active_prompt = get_prompt(ACTIVE_PROMPT_VERSION)

print(active_prompt)


You are a reliable AI assistant.

Use the supplied context to answer questions.
Use simple and clear English.
Do not invent information.
If the context does not contain the answer,
say that you do not have enough information.



# Creating the evaluation dataset

In [8]:
#Defining Test Cases
test_cases = [
    {
        "id": "test_001",
        "question": "What is RAG?",
        "context": "RAG combines document retrieval with language generation.",
        "expected_answer": "RAG retrieves relevant information and uses it to generate answers.",
        "expected_keywords": ["retrieves", "information", "generate"]
    },
    {
        "id": "test_002",
        "question": "What are embeddings?",
        "context": "Embeddings represent text as numerical vectors.",
        "expected_answer": "Embeddings convert text into numerical vectors.",
        "expected_keywords": ["text", "numerical", "vectors"]
    },
    {
        "id": "test_003",
        "question": "What is a vector database?",
        "context": "Vector databases store embeddings and support similarity search.",
        "expected_answer": "A vector database stores embeddings and supports similarity search.",
        "expected_keywords": ["embeddings", "similarity", "search"]
    },
    {
        "id": "test_004",
        "question": "Who invented Python?",
        "context": "The supplied context contains no information about the creator of Python.",
        "expected_answer": "The context does not provide that information.",
        "expected_keywords": ["context", "information"]
    }
]

In [9]:
#Saving the dataset
DATASET_FILE = DATASET_DIR / "test_cases.json"

with open(DATASET_FILE, "w", encoding="utf-8") as file:
    json.dump(test_cases, file, indent=4)

print("Evaluation dataset saved.")

Evaluation dataset saved.


# Building the LLM application

In [10]:
#Creating the answer generation function
def generate_answer(question, context, prompt):

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=prompt,
        input=f"""
        Context:
        {context}

        Question:
        {question}
        """,
        max_output_tokens=200
    )

    return response

In [11]:
#Test One request
sample = test_cases[0]

response = generate_answer(
    sample["question"],
    sample["context"],
    active_prompt
)

print(response.output_text)

RAG is a method that combines document retrieval with language generation.


# Building the evaluation engine 

In [12]:
#Implement Keyword coverage
def calculate_keyword_coverage(answer, expected_keywords):

    answer = answer.lower()

    if not expected_keywords:
        return 0.0

    matched = []

    for keyword in expected_keywords:
        if keyword.lower() in answer:
            matched.append(keyword)

    score = len(matched) / len(expected_keywords)

    return score

In [13]:
answer = "RAG retrieves relevant information and generates answers."

keywords = ["retrieves", "information", "generates"]

print(calculate_keyword_coverage(answer, keywords))

1.0


In [15]:
#Implement answer evaluation
def evaluate_answer(answer, test_case):

    score = calculate_keyword_coverage(
        answer,
        test_case["expected_keywords"]
    )

    passed = score >= 0.67

    return {
        "score": round(score, 3),
        "passed": passed
    }

# Adding observability

In [16]:
#Conifguring a log file
LOG_FILE = RESULT_DIR / "pipeline.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("llmops_pipeline")

logger.info("LLMOps pipeline initialized.")

2026-11-10 23:24:03,863 | INFO | LLMOps pipeline initialized.


In [17]:
#Create the complete test runner
def run_test_case(test_case, prompt, prompt_version):

    request_id = str(uuid.uuid4())

    start_time = time.perf_counter()

    logger.info(
        "Request started: %s",
        request_id
    )

    try:

        response = generate_answer(
            test_case["question"],
            test_case["context"],
            prompt
        )

        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        answer = response.output_text

        usage = response.usage

        input_tokens = usage.input_tokens if usage else 0
        output_tokens = usage.output_tokens if usage else 0

        evaluation = evaluate_answer(
            answer,
            test_case
        )

        result = {
            "request_id": request_id,
            "test_case_id": test_case["id"],
            "prompt_version": prompt_version,
            "model": "gpt-4.1-mini",
            "question": test_case["question"],
            "answer": answer,
            "score": evaluation["score"],
            "passed": evaluation["passed"],
            "latency_ms": round(latency_ms, 2),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "status": "success"
        }

        logger.info(
            "Request completed: %s",
            request_id
        )

        return result

    except Exception as error:

        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        logger.exception(
            "Request failed: %s",
            request_id
        )

        return {
            "request_id": request_id,
            "test_case_id": test_case["id"],
            "prompt_version": prompt_version,
            "latency_ms": round(latency_ms, 2),
            "status": "error",
            "error_type": type(error).__name__
        }

In [18]:
#Runnning the complete evaluation function
def run_evaluation_pipeline(
    test_cases,
    prompt_version
):

    prompt = get_prompt(prompt_version)

    results = []

    for test_case in test_cases:

        result = run_test_case(
            test_case,
            prompt,
            prompt_version
        )

        results.append(result)

    return results

In [19]:
#running version2 
results_v2 = run_evaluation_pipeline(
    test_cases,
    "v2"
)

print(json.dumps(results_v2, indent=4))


2026-11-10 23:25:35,824 | INFO | Request started: fb3b857f-9947-4fc1-a73b-d800e6e444f9
2026-11-10 23:25:37,739 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-10 23:25:37,745 | INFO | Request completed: fb3b857f-9947-4fc1-a73b-d800e6e444f9
2026-11-10 23:25:37,750 | INFO | Request started: 6f5fe646-8e58-4415-8e3b-2bbd66e11c32
2026-11-10 23:25:39,968 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-10 23:25:39,976 | INFO | Request completed: 6f5fe646-8e58-4415-8e3b-2bbd66e11c32
2026-11-10 23:25:40,020 | INFO | Request started: 71e9e06f-9cb1-4126-bb8a-2a789caba35f
2026-11-10 23:25:41,698 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-10 23:25:41,703 | INFO | Request completed: 71e9e06f-9cb1-4126-bb8a-2a789caba35f
2026-11-10 23:25:41,711 | INFO | Request started: 62baa985-6794-4802-a495-29d343e94661
2026-11-10 23:25:43,929 | INFO | HTTP Request: POST https://api.ope

[
    {
        "request_id": "fb3b857f-9947-4fc1-a73b-d800e6e444f9",
        "test_case_id": "test_001",
        "prompt_version": "v2",
        "model": "gpt-4.1-mini",
        "question": "What is RAG?",
        "answer": "RAG is a method that combines document retrieval with language generation.",
        "score": 0.0,
        "passed": false,
        "latency_ms": 1922.23,
        "input_tokens": 80,
        "output_tokens": 14,
        "status": "success"
    },
    {
        "request_id": "6f5fe646-8e58-4415-8e3b-2bbd66e11c32",
        "test_case_id": "test_002",
        "prompt_version": "v2",
        "model": "gpt-4.1-mini",
        "question": "What are embeddings?",
        "answer": "Embeddings are numerical vectors that represent text.",
        "score": 1.0,
        "passed": true,
        "latency_ms": 2227.11,
        "input_tokens": 78,
        "output_tokens": 10,
        "status": "success"
    },
    {
        "request_id": "71e9e06f-9cb1-4126-bb8a-2a789caba35f",
  

In [20]:
results_v1 = run_evaluation_pipeline(
    test_cases,
    "v1"
)

2026-11-10 23:25:46,443 | INFO | Request started: 4ad8c253-6992-4b93-9e34-949ab1ad1b43
2026-11-10 23:25:47,993 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-10 23:25:48,002 | INFO | Request completed: 4ad8c253-6992-4b93-9e34-949ab1ad1b43
2026-11-10 23:25:48,031 | INFO | Request started: e273a69d-dc7e-4bfb-aa26-ee9788a5b661
2026-11-10 23:25:50,554 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-10 23:25:50,563 | INFO | Request completed: e273a69d-dc7e-4bfb-aa26-ee9788a5b661
2026-11-10 23:25:50,564 | INFO | Request started: 3de68d74-b918-486b-8649-4de70a7f1159
2026-11-10 23:25:54,334 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-10 23:25:54,383 | INFO | Request completed: 3de68d74-b918-486b-8649-4de70a7f1159
2026-11-10 23:25:54,396 | INFO | Request started: b02aa53a-d252-4f80-af17-ce3d4757b032
2026-11-10 23:25:55,862 | INFO | HTTP Request: POST https://api.ope

# Creating an experiment summary||

In [21]:
#Calculating summary metrics
def summarize_results(results):

    successful = [
        result
        for result in results
        if result["status"] == "success"
    ]

    failed = len(results) - len(successful)

    if not successful:
        return {
            "total_requests": len(results),
            "successful_requests": 0,
            "failed_requests": failed
        }

    average_score = sum(
        result["score"] for result in successful
    ) / len(successful)

    pass_rate = sum(
        result["passed"] for result in successful
    ) / len(successful)

    average_latency = sum(
        result["latency_ms"] for result in successful
    ) / len(successful)

    total_input_tokens = sum(
        result["input_tokens"] for result in successful
    )

    total_output_tokens = sum(
        result["output_tokens"] for result in successful
    )

    return {
        "total_requests": len(results),
        "successful_requests": len(successful),
        "failed_requests": failed,
        "average_keyword_coverage": round(average_score, 3),
        "pass_rate": round(pass_rate, 3),
        "average_latency_ms": round(average_latency, 2),
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens
    }

In [22]:
#Generating the resport
summary_v2 = summarize_results(results_v2)

print("EVALUATION REPORT")
print(json.dumps(summary_v2, indent=4))

EVALUATION REPORT
{
    "total_requests": 4,
    "successful_requests": 4,
    "failed_requests": 0,
    "average_keyword_coverage": 0.75,
    "pass_rate": 0.75,
    "average_latency_ms": 2016.1,
    "total_input_tokens": 321,
    "total_output_tokens": 76
}


In [23]:
#Save JSON results
RESULTS_FILE = RESULT_DIR / "evaluation_results.json"

experiment_report = {
    "experiment_id": str(uuid.uuid4()),
    "prompt_version": "v2",
    "model": "gpt-4.1-mini",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "summary": summary_v2,
    "results": results_v2
}

with open(RESULTS_FILE, "w", encoding="utf-8") as file:

    json.dump(
        experiment_report,
        file,
        indent=4
    )

print("JSON report saved.")

JSON report saved.


In [24]:
#Export CSV
df = pd.DataFrame(results_v2)

CSV_FILE = RESULT_DIR / "evaluation_results.csv"

df.to_csv(CSV_FILE, index=False)

print("CSV report saved.")

df

CSV report saved.


,request_id,test_case_id,prompt_version,model,question,answer,score,passed,latency_ms,input_tokens,output_tokens,status
0,fb3b857f-9947-4fc1-a73b-d800e6e444f9,test_001,v2,gpt-4.1-mini,What is RAG?,RAG is a method that combines document retriev...,0.0,False,1922.23,80,14,success
1,6f5fe646-8e58-4415-8e3b-2bbd66e11c32,test_002,v2,gpt-4.1-mini,What are embeddings?,Embeddings are numerical vectors that represen...,1.0,True,2227.11,78,10,success
2,71e9e06f-9cb1-4126-bb8a-2a789caba35f,test_003,v2,gpt-4.1-mini,What is a vector database?,A vector database is a database that stores em...,1.0,True,1683.37,81,40,success
3,62baa985-6794-4802-a495-29d343e94661,test_004,v2,gpt-4.1-mini,Who invented Python?,The context does not contain information about...,1.0,True,2231.69,82,12,success


In [25]:
#Comparing Prompt Versions
summary_v1 = summarize_results(results_v1)
summary_v2 = summarize_results(results_v2)

comparison = pd.DataFrame([
    {"version": "v1", **summary_v1},
    {"version": "v2", **summary_v2}
])

comparison

,version,total_requests,successful_requests,failed_requests,average_keyword_coverage,pass_rate,average_latency_ms,total_input_tokens,total_output_tokens
0,v1,4,4,0,0.667,0.50,2345.73,193,197
1,v2,4,4,0,0.750,0.75,2016.10,321,76
